# JS09 Lab 4: ANNOY vs FAISS vs HNSW

Jobsheet 09, Lab 4. Ketiga pustaka ANN dijalankan pada dataset yang sama berukuran 1 juta baris dengan 5 dimensi. Yang dibandingkan adalah waktu membangun indeks dan waktu menjawab satu query.

Jumlah data 1 juta baris cukup berat. Kalau memori terbatas, angka `n_data` bisa diturunkan menjadi `100_000` seperti saran pada modul.

In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# Build 1mio rows with 5D
# ===============================
n_data = 1_000_000   # try 100_000 if you have limited memory
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===


Build time: 4.583094120025635 detik
Query time: 0.009661674499511719 detik
Neighbors: [474211, 272434, 629627, 917559, 533432] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.007054567337036133 detik
Query time: 0.005430459976196289 detik
Neighbors: [474211 272434 629627 917559 533432] ...

=== HNSW (hnswlib) ===


Build time: 33.88785910606384 detik
Query time: 0.18460392951965332 detik
Neighbors: [474211 272434 629627 917559 533432] ...


Hasil bisa berbeda antar mesin. Yang penting adalah polanya: FAISS Flat paling cepat dibangun karena tidak membangun struktur tambahan, ANNOY dan HNSW lebih lama dibangun tetapi query-nya jauh lebih cepat. Ketiganya menemukan tetangga yang sama pada data acak 5D ini.